# Этап 1. Сбор и обработка данных

Ноутбук идёт **строго по методичке AITU**: табл. 7 «Data collection» и табл. 8 «Data processing».
Номер раздела = номер шага методички. Весь код — в `src/` (`src/pipeline.py` связывает шаги),
ноутбук только вызывает шаги по очереди и показывает результат.

Результаты сохраняются на Drive в `thesis/outputs/stage1_data/`:
`quality/` (проблемы данных), `tables/` (CSV), `figures/` (PNG, 300 dpi), `report.html` и `report.md`.

То же самое одной командой, без ноутбука: `python -m src.pipeline --thesis /content/drive/MyDrive/thesis`

## 0. Настройка окружения

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    token = userdata.get('GH_TOKEN')
    !git clone -q https://{token}@github.com/zhannaa0/kazakh-hybrid-nlp.git /content/project 2>/dev/null || git -C /content/project pull -q
    %cd /content/project
    !pip install -q -r requirements.txt
    PROJECT = Path('/content/project')
    THESIS = Path('/content/drive/MyDrive/thesis')
else:                                   # запуск на своём компьютере
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    THESIS = Path(os.environ.get('THESIS_DIR', PROJECT.parent / 'thesis'))
sys.path.insert(0, str(PROJECT))

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image, Markdown

from src import pipeline as P
from src.data.collect import SOURCES

pd.set_option('display.float_format', '{:,.3f}'.format)
pd.set_option('display.max_colwidth', 80)
ctx = P.Context(THESIS)
print('исходные данные :', ctx.raw)
print('результаты      :', ctx.out)

# Таблица 7. Data collection

### 7.1 Defining a data type
Тексты на казахском с **категориальной разметкой каждого токена**: теги сущностей IOB2 (25 типов),
части речи UPOS (17), морфологические признаки UD FEATS. Вход качественный (текст), в анализе
превращается в количественные показатели: частоты меток, F1, accuracy.

### 7.2 Careful planning
План из этапов 0–5 (данные → rule-based → нейросеть → гибрид → сравнение). Ограничения: нет бюджета
на новую разметку, бесплатный Colab (GPU T4, лимит сессий), работа по вечерам. Отсюда — готовые
корпуса, один общий трансформер, обработка данных на CPU.

### 7.3 Selecting data collection methods
**Вторичные данные — эталонные размеченные корпуса.** Плюсы: экспертная разметка, сравнимость с
опубликованными результатами, бесплатно. Минусы: фиксированные схемы разметки, жанр (новости,
госсайты), малый UD-KTB, пересечение train/test в KazNERD. Отклонено: собственная разметка
(время, стоимость), «серебряные» данные вроде WikiANN (шумная разметка).

### 7.4 Development of tools
Вместо анкеты инструмент исследования — код: конвейер данных (`src/data`), единый скрипт оценки
для всех 9 комбинаций задача × метод с фиксированными метриками (`src/eval`), автотесты (`tests`).

### 7.5 Data collection
Источники и **зафиксированные версии** (`src/data/collect.py`). Каждый файл сверяется с эталонным
SHA-256: если файл на Drive отличается от того, на котором получены результаты, шаг остановится.
Чтобы скачать недостающие файлы автоматически: `P.step_collect(ctx, download=True)`.

In [ ]:
display(pd.DataFrame([{k: s[k] for k in ('name', 'version', 'license', 'repo', 'citation')}
                      for s in SOURCES.values()]))
P.step_collect(ctx, download=False)

### 7.6–7.8 Processing, analysis, documentation
Подробно — в таблице 8 ниже. Анализ **результатов моделей** (7.7) появится, когда методы
начнут сохранять предсказания: `python -m src.eval.evaluate results/preds --significance`.

# Таблица 8. Data processing

### 8.1 Data quality check
Проверяем **исходные** файлы до любых изменений: ошибки формата, пропуски, несогласованности.
Однозначные ошибки исправляются на шаге 8.3; остальное только фиксируется (официальные сплиты
должны остаться сравнимыми со статьями). Полные списки — в `outputs/stage1_data/quality/`.

In [ ]:
P.step_quality(ctx)

In [ ]:
print('Кандидаты в пропуски разметки NER (для ручной проверки):')
display(ctx.r['quality']['gaps'].head(10))
print('Противоречивая разметка одинаковых предложений:')
ctx.r['quality']['conflicts'].head(5)

### 8.2 Coding and structuring · 8.3 Data transformation
- **Структура:** оба корпуса → единый JSONL (`id`, `tokens`, метки по токенам); многословные
  токены UD → синтаксические слова.
- **Кодирование:** метки → числа, словари строятся только по train (`label_maps.json`).
  Для FEATS два варианта: строка целиком или каждая категория отдельно.
- **Преобразования:** нормализация Unicode NFC; латинские буквы-двойники → кириллица;
  `I-X` без начала → `B-X`. Стандартизация и логарифм к категориальным меткам неприменимы.
- **Разбиения:** KazNERD — официальный сплит; UD-KTB — новый 80/10/10, стратифицированный
  по исходному тексту, seed 42.

In [ ]:
manifest = P.step_prepare(ctx, seed=42)
display(pd.Series(manifest['label_space']).to_frame('число меток'))
display(pd.Series(manifest['cleaning']).to_frame('исправлений'))
ctx.r['cleaning_log'].sample(8, random_state=0)[['id', 'rule', 'before', 'after']]

In [ ]:
# Как выглядит одно предложение в едином формате
s = ctx.r['udktb']['train'][0]
print(json.dumps({k: (v[:6] if isinstance(v, list) else v) for k, v in s.items()}, ensure_ascii=False, indent=1))

### 8.4 Aggregation and summation · 8.5 Statistical analysis
Агрегирование по сплитам, типам сущностей, частям речи, категориям признаков, исходным текстам,
длине предложений и частотным диапазонам; описательная статистика (доли, дисбаланс, энтропия,
выбросы по IQR, OOV). Все таблицы сохраняются в `tables/`.

In [ ]:
P.step_analyse(ctx)
t = ctx.r['tables']
for name in ('split_sizes', 'class_imbalance', 'descriptive_statistics', 'sentence_length_outliers'):
    display(Markdown(f'**{name}**')); display(t[name])

In [ ]:
display(Markdown('**OOV и редкие слова**'))
display(pd.concat({'KazNERD': t['vocabulary_kaznerd'], 'UD-KTB': t['vocabulary_udktb']}))
display(Markdown('**Веса классов NER для функции потерь (если будем компенсировать дисбаланс)**'))
t['ner_class_weights'].sort_values('train_entities').head(8)

### 8.6 Data visualisation
Данные **до и после** предобработки (рис. 1–3) и разведочный анализ (рис. 4–10).
PNG 300 dpi в `figures/` — готовы для диссертации.

In [ ]:
figs = P.step_visualise(ctx)
for i, (name, caption) in enumerate(figs.items(), 1):
    display(Markdown(f'**Figure {i}.** {caption}'))
    display(Image(filename=str(ctx.figures / f'{name}.png'), width=760))

### 8.7 Interpretation of results
Для данных — выводы в конце ноутбука. Для результатов моделей: каждый исследовательский вопрос
решается **внутри одной задачи** на одном тесте; приводится разница с 95% ДИ и p-value с поправкой
Холма (`src/eval/significance.py`), а также разбивка ошибок по OOV, длине и источнику
(`src/eval/breakdown.py`).

### 8.8 Creating a report
Отчёт собирается из результатов, поэтому цифры в нём всегда актуальны.

In [ ]:
path = P.step_report(ctx)
print('HTML (для просмотра):', path)
print('Markdown (для диссертации):', ctx.out / 'report.md')

### 8.9 Secrecy and confidentiality · 8.10 Documentation
Персональные данные не собираются; корпуса открытые (CC BY 4.0, CC BY-SA 4.0), используются со ссылкой
на авторов. Данные — на Google Drive, не в git; код — в приватном репозитории; токен — в Colab Secrets.
Журнал запуска: версии, лицензии, хэши, время каждого шага.

In [ ]:
log = P.step_document(ctx, timings=[])
print(json.dumps({k: log[k] for k in ('finished_at', 'code_commit', 'sources', 'storage')}, ensure_ascii=False, indent=1))

### 8.11 Publication of results
Диссертация; после защиты — код и номера разбиений (не сами данные) в открытом репозитории;
по согласованию с руководителем — короткая статья.

---

## Выводы по данным

1. **Масштаб:** KazNERD — 112 702 предложения (≈1,3 млн токенов), UD-KTB — 1 078 (≈10,5 тыс. слов):
   разница на два порядка, задуманный контраст «больше / мало ресурсов».
2. **Качество высокое:** ошибок формата нет; исправлено 76 латинских букв-двойников и 16 тегов IOB2.
   Предложения не удалялись.
3. **Сплит UD-KTB пересобран:** официальный train — 31 предложение из других текстов, чем test;
   новый 864 / 107 / 107 с одинаковым жанровым составом.
4. **Дисбаланс:** CARDINAL, DATE и GPE ≈ 53% сущностей; NON_HUMAN — 6 в train (≈3900:1). Поэтому
   кроме micro-F1 — macro-F1 и F1 по классам.
5. **Редкие слова — главная трудность UD-KTB:** 41–45% словоупотреблений dev/test нет в train
   (KazNERD — 4,8%). Это прямая мотивация морфологических правил в гибриде.
6. **Морфология:** 319 наборов FEATS, треть — единичные; неизвестных наборов в test ≈1,6%.
7. **KazNERD:** 2,7% test есть в train, 19 противоречий, кандидаты в пропуски сущностей —
   официальный сплит сохранён, эти цифры указываются в главе о данных.